# Packet P-013 — Temporal / Publication Bias Check

**Agent OS v3.0 — Materia Arche**

## Decision Question
> *Are our model predictions generalizable across data sources, or do they exploit batch/publication artifacts?*

### Motivation
The Perovskite Database Project aggregates measurements from many independent publications. If devices from the same paper share systematic biases (e.g., similar aging protocols, measurement setups, or compositional batches), a model trained on random splits could exploit within-paper correlations rather than learning genuine material-stability relationships.

**Method:** Leave-One-Group-Out (LOGO) cross-validation. We first check if `Ref_ID` forms usable publication groups. If each Ref_ID is unique (one device per paper entry), we fall back to proxy clustering: KMeans (k=20) on composition features (MA, FA, Cs, Pb, I, Br) to simulate "batch" groups. Train on all groups except one, predict on the held-out group, compute Kendall tau-b. Compare against standard random CV tau-b (0.289).

| Item | Value |
|---|---|
| Dataset | perovskite_stability_clean.csv (1,543 devices) |
| Features | 16 compositional + process + JV features |
| Target | log1p(Stability_PCE_T80) |
| Model | ExtraTreesRegressor (locked config) |
| Benchmark | Random CV tau-b = 0.289 |

In [1]:
"""Cell 2 — Load data and identify grouping variable."""
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.cluster import KMeans

# ── Load ──
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns\n")

# ── Check for publication/reference columns ──
ref_candidates = [c for c in df.columns if any(kw in c.lower() for kw in
                  ['ref', 'doi', 'author', 'paper', 'source', 'pub', 'batch'])]
print("Candidate grouping columns:", ref_candidates)

# ── Features & target ──
FEATURES = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
            'MA', 'FA', 'Cs',
            'first_Prvskt_annealing_temperature',
            'first_Prvskt_thermal_annealing_time',
            'Perovskite_thickness', 'Cell_area_measured',
            'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF']
TARGET = 'Stability_PCE_T80'

X = df[FEATURES].copy()
y = np.log1p(df[TARGET].values)
print(f"\nFeatures: {len(FEATURES)},  Target: log1p({TARGET})")
print(f"y range: [{y.min():.2f}, {y.max():.2f}]")

# ── Determine grouping ──
# Check if Ref_ID forms actual groups (multiple devices per Ref_ID)
USE_REF_ID = False
if 'Ref_ID' in df.columns and df['Ref_ID'].notna().sum() > 100:
    ref_counts = df['Ref_ID'].value_counts()
    n_groups_with_20 = (ref_counts >= 20).sum()
    n_groups_with_5 = (ref_counts >= 5).sum()
    print(f"\nRef_ID analysis:")
    print(f"  Unique Ref_IDs: {df['Ref_ID'].nunique()}")
    print(f"  Groups with >= 20 devices: {n_groups_with_20}")
    print(f"  Groups with >= 5 devices:  {n_groups_with_5}")
    print(f"  Max group size: {ref_counts.max()}")
    print(f"  Top 10 Ref_IDs by count:")
    print(f"  {ref_counts.head(10).to_string()}")
    if n_groups_with_20 >= 3:
        USE_REF_ID = True
        groups = df['Ref_ID'].values
        group_label = "Ref_ID (publication reference)"
        print(f"\n>>> Using Ref_ID as grouping variable")
    else:
        print(f"\n>>> Ref_ID exists but each ID is nearly unique — not useful for LOGO")

if not USE_REF_ID:
    # Proxy: cluster on composition + process features
    comp_cols = ['MA', 'FA', 'Cs', 'Pb', 'I', 'Br']
    print(f"\n>>> Using proxy clustering on composition: {comp_cols}")
    km = KMeans(n_clusters=20, random_state=42, n_init=10)
    groups = km.fit_predict(df[comp_cols].fillna(0))
    group_label = "KMeans proxy cluster (k=20 on composition)"
    print(f"    Created 20 proxy groups via KMeans")

print(f"    Group label: {group_label}")

Dataset: 1543 rows, 47 columns

Candidate grouping columns: ['Ref_ID']

Features: 16,  Target: log1p(Stability_PCE_T80)
y range: [0.00, 9.04]

Ref_ID analysis:
  Unique Ref_IDs: 1543
  Groups with >= 20 devices: 0
  Groups with >= 5 devices:  0
  Max group size: 1
  Top 10 Ref_IDs by count:
  Ref_ID
26       1
27694    1
29165    1
29162    1
29151    1
29149    1
29148    1
29146    1
29139    1
29113    1

>>> Ref_ID exists but each ID is nearly unique — not useful for LOGO

>>> Using proxy clustering on composition: ['MA', 'FA', 'Cs', 'Pb', 'I', 'Br']


    Created 20 proxy groups via KMeans
    Group label: KMeans proxy cluster (k=20 on composition)


/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: invalid value encountered in matmul
  curr

In [2]:
"""Cell 3 — Leave-One-Group-Out (LOGO) cross-validation."""

ET_PARAMS = dict(
    n_estimators=700, max_features='sqrt', min_samples_split=3,
    min_samples_leaf=1, bootstrap=False, random_state=42, n_jobs=-1
)

# ── Group sizes ──
group_series = pd.Series(groups)
group_counts = group_series.value_counts()
print(f"Total groups: {len(group_counts)}")
print(f"Groups with >= 20 devices: {(group_counts >= 20).sum()}")
print(f"Devices in those groups: {group_counts[group_counts >= 20].sum()}")
print(f"\nGroup size distribution:")
print(group_counts.describe())
print(f"\nTop 15 groups by size:")
print(group_counts.head(15).to_string())

# ── LOGO CV ──
MIN_GROUP_SIZE = 20
eligible_groups = group_counts[group_counts >= MIN_GROUP_SIZE].index.tolist()

logo_results = []
for g in eligible_groups:
    mask_test = (group_series == g).values
    mask_train = ~mask_test

    X_train, y_train = X.iloc[mask_train], y[mask_train]
    X_test, y_test = X.iloc[mask_test], y[mask_test]

    et = ExtraTreesRegressor(**ET_PARAMS)
    et.fit(X_train, y_train)
    y_pred = et.predict(X_test)

    tau, p = kendalltau(y_test, y_pred)
    logo_results.append({
        'group': g,
        'n_test': int(mask_test.sum()),
        'n_train': int(mask_train.sum()),
        'tau_b': tau,
        'p_value': p
    })

logo_df = pd.DataFrame(logo_results).sort_values('tau_b', ascending=False)
print(f"\n{'='*70}")
print(f"LOGO CV Results — {len(logo_df)} eligible groups (>= {MIN_GROUP_SIZE} devices)")
print(f"{'='*70}")
print(logo_df.to_string(index=False, float_format='%.4f'))

# ── Weighted-average tau-b (weighted by group test size) ──
weights = logo_df['n_test'].values
logo_tau_weighted = np.average(logo_df['tau_b'].values, weights=weights)
logo_tau_mean = logo_df['tau_b'].mean()
logo_tau_median = logo_df['tau_b'].median()

print(f"\n--- Summary ---")
print(f"Mean   LOGO tau-b: {logo_tau_mean:.4f}")
print(f"Median LOGO tau-b: {logo_tau_median:.4f}")
print(f"Weighted-avg LOGO tau-b (by group size): {logo_tau_weighted:.4f}")

Total groups: 20
Groups with >= 20 devices: 5
Devices in those groups: 1493

Group size distribution:
count     20.000000
mean      77.150000
std      228.097226
min        1.000000
25%        1.000000
50%        2.000000
75%       19.250000
max      960.000000
Name: count, dtype: float64

Top 15 groups by size:
7     960
0     425
15     61
16     27
12     20
11     19
5      10
14      4
19      3
9       2
2       2
4       2
13      1
3       1
17      1



LOGO CV Results — 5 eligible groups (>= 20 devices)
 group  n_test  n_train   tau_b  p_value
    16      27     1516  0.2575   0.0605
    15      61     1482  0.0992   0.2623
     7     960      583  0.0729   0.0008
    12      20     1523  0.0587   0.7204
     0     425     1118 -0.0043   0.8942

--- Summary ---
Mean   LOGO tau-b: 0.0968
Median LOGO tau-b: 0.0729
Weighted-avg LOGO tau-b (by group size): 0.0551


In [3]:
"""Cell 4 — Compare LOGO tau-b vs standard random CV tau-b."""

RANDOM_CV_TAU = 0.289

print("="*60)
print("COMPARISON: LOGO CV  vs  Random CV")
print("="*60)
print(f"Random CV tau-b (benchmark):      {RANDOM_CV_TAU:.4f}")
print(f"LOGO CV tau-b (weighted avg):     {logo_tau_weighted:.4f}")
print(f"LOGO CV tau-b (mean):             {logo_tau_mean:.4f}")
print(f"LOGO CV tau-b (median):           {logo_tau_median:.4f}")

gap = RANDOM_CV_TAU - logo_tau_weighted
ratio = logo_tau_weighted / RANDOM_CV_TAU if RANDOM_CV_TAU != 0 else float('inf')

print(f"\nGap (random - LOGO weighted):     {gap:.4f}")
print(f"Ratio (LOGO / random):            {ratio:.4f}  ({ratio*100:.1f}%)")

if ratio >= 0.80:
    print(f"\n>>> LOGO retains >= 80% of random CV performance.")
    print(f"    Interpretation: Generalization holds across publications.")
    print(f"    Batch/publication bias is NOT a major concern.")
elif ratio >= 0.50:
    print(f"\n>>> LOGO retains {ratio*100:.0f}% of random CV performance.")
    print(f"    Interpretation: Moderate batch effects present.")
    print(f"    Some within-paper correlation boosts random CV.")
else:
    print(f"\n>>> LOGO retains only {ratio*100:.0f}% of random CV performance.")
    print(f"    Interpretation: Severe batch dependency detected.")
    print(f"    Model may be exploiting within-paper correlations.")

# ── Per-group breakdown ──
n_positive = (logo_df['tau_b'] > 0).sum()
n_significant = (logo_df['p_value'] < 0.05).sum()
print(f"\nPer-group stats:")
print(f"  Groups with tau-b > 0: {n_positive}/{len(logo_df)}")
print(f"  Groups with p < 0.05:  {n_significant}/{len(logo_df)}")
print(f"  Worst group tau-b:     {logo_df['tau_b'].min():.4f}")
print(f"  Best  group tau-b:     {logo_df['tau_b'].max():.4f}")

COMPARISON: LOGO CV  vs  Random CV
Random CV tau-b (benchmark):      0.2890
LOGO CV tau-b (weighted avg):     0.0551
LOGO CV tau-b (mean):             0.0968
LOGO CV tau-b (median):           0.0729

Gap (random - LOGO weighted):     0.2339
Ratio (LOGO / random):            0.1907  (19.1%)

>>> LOGO retains only 19% of random CV performance.
    Interpretation: Severe batch dependency detected.
    Model may be exploiting within-paper correlations.

Per-group stats:
  Groups with tau-b > 0: 4/5
  Groups with p < 0.05:  1/5
  Worst group tau-b:     -0.0043
  Best  group tau-b:     0.2575


In [4]:
"""Cell 5 — Panel device group membership check."""

PANEL_INDICES = [850, 1320, 119]
PANEL_LABELS = {
    850:  "Device 850  (MA-only, T80=3400h)",
    1320: "Device 1320 (MA-FA mixed, T80=940h)",
    119:  "Device 119  (FA-Cs, T80=3423h)"
}

print("="*60)
print("PANEL DEVICE — GROUP MEMBERSHIP CHECK")
print("="*60)

for idx in PANEL_INDICES:
    if idx >= len(df):
        print(f"\n{PANEL_LABELS[idx]}: index {idx} out of range (dataset has {len(df)} rows)")
        continue
    g = groups[idx] if hasattr(groups, '__getitem__') else group_series.iloc[idx]
    g_size = (group_series == g).sum()
    g_pct = g_size / len(df) * 100

    # Check if this group was in LOGO and its tau-b
    group_logo = logo_df[logo_df['group'] == g]
    if len(group_logo) > 0:
        g_tau = group_logo['tau_b'].values[0]
        tau_str = f"tau-b = {g_tau:.4f}"
    else:
        tau_str = f"group too small for LOGO (< {MIN_GROUP_SIZE} devices)"

    print(f"\n{PANEL_LABELS[idx]}")
    print(f"  Group: {g}")
    print(f"  Group size: {g_size} devices ({g_pct:.1f}% of dataset)")
    print(f"  LOGO result: {tau_str}")

# ── Check if panel devices are concentrated in dominant groups ──
print(f"\n--- Dominance check ---")
panel_groups = [groups[idx] for idx in PANEL_INDICES if idx < len(df)]
unique_panel_groups = set(panel_groups)
print(f"Panel devices span {len(unique_panel_groups)} distinct group(s)")
if len(unique_panel_groups) == 1:
    print("WARNING: All panel devices come from the SAME group!")
    print("Rankings may reflect within-group bias rather than true generalization.")
elif len(unique_panel_groups) == len(PANEL_INDICES):
    print("Panel devices are spread across different groups — good diversity.")

PANEL DEVICE — GROUP MEMBERSHIP CHECK

Device 850  (MA-only, T80=3400h)
  Group: 11
  Group size: 19 devices (1.2% of dataset)
  LOGO result: group too small for LOGO (< 20 devices)

Device 1320 (MA-FA mixed, T80=940h)
  Group: 0
  Group size: 425 devices (27.5% of dataset)
  LOGO result: tau-b = -0.0043

Device 119  (FA-Cs, T80=3423h)
  Group: 15
  Group size: 61 devices (4.0% of dataset)
  LOGO result: tau-b = 0.0992

--- Dominance check ---
Panel devices span 3 distinct group(s)
Panel devices are spread across different groups — good diversity.


In [5]:
"""Cell 6 — Save per-group results to CSV."""

out = logo_df.copy()
out['group_label'] = group_label
out['random_cv_tau'] = RANDOM_CV_TAU
out['logo_weighted_avg_tau'] = logo_tau_weighted
out['ratio_logo_vs_random'] = ratio

out_path = "Packet_P013_Publication_Bias.csv"
out.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"  Rows: {len(out)}")
print(f"  Columns: {list(out.columns)}")

Saved: Packet_P013_Publication_Bias.csv
  Rows: 5
  Columns: ['group', 'n_test', 'n_train', 'tau_b', 'p_value', 'group_label', 'random_cv_tau', 'logo_weighted_avg_tau', 'ratio_logo_vs_random']


In [6]:
"""Cell 7 — Honest status footer."""

print("="*70)
print("PACKET P-013 — TEMPORAL / PUBLICATION BIAS CHECK")
print("="*70)
print(f"\nGrouping variable: {group_label}")
print(f"Eligible groups (>= {MIN_GROUP_SIZE} devices): {len(logo_df)}")
print(f"Devices covered: {logo_df['n_test'].sum()} / {len(df)}")
print(f"\nRandom CV tau-b:           {RANDOM_CV_TAU:.4f}")
print(f"LOGO CV tau-b (weighted):  {logo_tau_weighted:.4f}")
print(f"LOGO CV tau-b (mean):      {logo_tau_mean:.4f}")
print(f"LOGO CV tau-b (median):    {logo_tau_median:.4f}")
print(f"Ratio (LOGO / random):     {ratio:.4f}  ({ratio*100:.1f}%)")

# ── Verdict ──
print(f"\n{'─'*70}")
if ratio >= 0.80:
    STATUS = "CONFIRMED"
    print(f"STATUS: *** {STATUS} ***")
    print(f"LOGO tau-b >= 80% of random CV tau-b.")
    print(f"Model generalization holds across publication/data-source boundaries.")
    print(f"No significant publication bias detected.")
elif ratio >= 0.50:
    STATUS = "PROMISING"
    print(f"STATUS: *** {STATUS} ***")
    print(f"LOGO tau-b >= 50% of random CV tau-b.")
    print(f"Moderate batch effects present — some within-paper correlation")
    print(f"boosts random CV, but the model still captures real signal.")
else:
    STATUS = "NEGATIVE"
    print(f"STATUS: *** {STATUS} ***")
    print(f"LOGO tau-b < 50% of random CV tau-b.")
    print(f"Severe batch dependency — model may be exploiting")
    print(f"within-paper correlations rather than genuine material relationships.")

print(f"\nLimitations:")
if USE_REF_ID:
    print(f"  - Ref_ID identifies papers but not necessarily independent labs")
    print(f"  - Small groups (< {MIN_GROUP_SIZE} devices) excluded from LOGO")
else:
    print(f"  - No true publication identifier available; used proxy clustering")
    print(f"  - Proxy groups may not align with actual paper boundaries")
print(f"  - LOGO is conservative: some cross-paper generalization loss is expected")
print(f"  - tau-b on small groups has high variance")
print(f"{'─'*70}")
print(f"\nPacket P-013 complete.  Status: {STATUS}")

PACKET P-013 — TEMPORAL / PUBLICATION BIAS CHECK

Grouping variable: KMeans proxy cluster (k=20 on composition)
Eligible groups (>= 20 devices): 5
Devices covered: 1493 / 1543

Random CV tau-b:           0.2890
LOGO CV tau-b (weighted):  0.0551
LOGO CV tau-b (mean):      0.0968
LOGO CV tau-b (median):    0.0729
Ratio (LOGO / random):     0.1907  (19.1%)

──────────────────────────────────────────────────────────────────────
STATUS: *** NEGATIVE ***
LOGO tau-b < 50% of random CV tau-b.
Severe batch dependency — model may be exploiting
within-paper correlations rather than genuine material relationships.

Limitations:
  - No true publication identifier available; used proxy clustering
  - Proxy groups may not align with actual paper boundaries
  - LOGO is conservative: some cross-paper generalization loss is expected
  - tau-b on small groups has high variance
──────────────────────────────────────────────────────────────────────

Packet P-013 complete.  Status: NEGATIVE
